# Notebook 08 — Train Baseline Models

## Goal
Train two simple baseline models that all more complex models must beat:
1. **Naive baseline** — predicts next month = last observed MoM change
2. **ARIMA** — time series model on PAYEMS MoM change

All training, prediction, and evaluation logic is written here. No black-box scripts.

---

## What can go wrong
- ARIMA may fail to converge for some orders — try simpler orders if needed
- COVID outliers (2020-03 to 2020-05) will dominate RMSE — inspect separately
- If test set is small (<24 months), metrics have wide confidence intervals

---

## What you must inspect
- Prediction sample table: do predictions look plausible?
- Are metrics reasonable? (Naive MAE on payroll data is typically 80–200K)
- Does ARIMA actually beat the naive baseline? (it often does not)

In [ ]:
import pathlib, json, datetime, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml
import optuna
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/labor_news_rag_project')
REPO_DIR = pathlib.Path('/content/economic-news-labor-rag')
METRICS_DIR = DRIVE_ROOT / 'outputs' / 'metrics'

MODEL_READY_PATH = DRIVE_ROOT / 'data' / 'model_ready' / 'labor_forecasting_dataset_approved.parquet'

ap_path = DRIVE_ROOT / 'approvals' / '07_model_ready_dataset_approved.json'
if not ap_path.exists():
    raise FileNotFoundError('STOP: Notebook 07 not approved.')
with open(ap_path) as f:
    ap = json.load(f)
assert ap['approved'], 'NB 07 not approved.'

with open(REPO_DIR / 'configs' / 'base.yaml') as f:
    base_cfg = yaml.safe_load(f)
with open(REPO_DIR / 'configs' / 'model_config.yaml') as f:
    model_cfg = yaml.safe_load(f)

_splits_key = 'test_splits' if base_cfg.get('test_mode', False) else 'splits'
_splits = model_cfg[_splits_key]
TRAIN_END = pd.Timestamp(_splits['train_end'])
VAL_END   = pd.Timestamp(_splits['validation_end'])
TEST_START = pd.Timestamp(_splits['test_start'])
TARGET = model_cfg['target_column']
SEED = 42

MODE = 'TEST (quick run)' if base_cfg.get('test_mode', False) else 'FULL (research run)'
print(f'Mode   : {MODE}')
print(f'Splits : train≤{_splits["train_end"]} | val≤{_splits["validation_end"]} | test≥{_splits["test_start"]}')

model_df = pd.read_parquet(MODEL_READY_PATH)
model_df['forecast_date'] = pd.to_datetime(model_df['forecast_date'])
model_df = model_df.dropna(subset=[TARGET]).sort_values('forecast_date').reset_index(drop=True)

train_df = model_df[model_df['forecast_date'] <= TRAIN_END]
val_df   = model_df[(model_df['forecast_date'] > TRAIN_END) & (model_df['forecast_date'] <= VAL_END)]
test_df  = model_df[model_df['forecast_date'] >= TEST_START]

print(f'Train: {len(train_df)} rows | Val: {len(val_df)} rows | Test: {len(test_df)} rows')

## Metric functions

In [ ]:
def compute_metrics(actuals, predictions, model_name, period='test'):
    actuals = np.array(actuals)
    predictions = np.array(predictions)
    mask = ~(np.isnan(actuals) | np.isnan(predictions))
    actuals, predictions = actuals[mask], predictions[mask]
    mae = mean_absolute_error(actuals, predictions)
    rmse = np.sqrt(mean_squared_error(actuals, predictions))
    da = np.mean(np.sign(actuals) == np.sign(predictions))
    return {
        'model': model_name, 'period': period, 'n': len(actuals),
        'MAE': round(mae, 2), 'RMSE': round(rmse, 2), 'DA': round(da, 4)
    }

def show_prediction_sample(dates, actuals, predictions, model_name, n=10):
    df = pd.DataFrame({
        'forecast_date': dates,
        'actual': actuals,
        'prediction': predictions,
        'error': np.array(actuals) - np.array(predictions),
    })
    print(f'\n=== {model_name}: Prediction Sample (last {n} rows) ===')
    print(df.tail(n).to_string(index=False))
    return df

all_metrics = []
print('Metric functions defined.')

## Hyperparameter Search — Naive Lag Weighting

Grid search over how much weight to put on `payems_lag1` vs `payems_lag2`.

- `lag1_weight = 1.0` → pure last-value (original naive)
- `lag1_weight = 0.5` → mean of last 2 months
- `lag1_weight = 0.0` → use only lag2

Optimizes **validation MAE** — test set untouched.

In [ ]:
naive_ps = model_cfg['models']['naive'].get('param_search', {})
weights = naive_ps.get('search_space', {}).get('lag1_weight', [1.0, 0.9, 0.7, 0.5, 0.3, 0.0])

print('Naive lag weight grid search (val MAE):')
naive_results = []
for w in weights:
    lag1 = val_df['payems_lag1'].fillna(0)
    lag2 = val_df['payems_lag2'].fillna(val_df['payems_lag1'].fillna(0))
    preds = w * lag1 + (1 - w) * lag2
    mask = val_df[TARGET].notna()
    mae = mean_absolute_error(val_df.loc[mask, TARGET], preds[mask])
    naive_results.append({'lag1_weight': w, 'val_mae': round(mae, 2)})
    print(f'  weight={w:.1f}  (lag1 * {w:.1f} + lag2 * {1-w:.1f})  →  val MAE = {mae:.2f}K')

best_naive = min(naive_results, key=lambda x: x['val_mae'])
BEST_NAIVE_WEIGHT = best_naive['lag1_weight']
print(f'\nBest weight: {BEST_NAIVE_WEIGHT} (val MAE = {best_naive["val_mae"]:.2f}K)')

## Hyperparameter Search — ARIMA Order (p, d, q)

Optuna TPE search over ARIMA orders. Each trial fits the model on the training set and evaluates with a **walk-forward** prediction on the validation set.

- p (AR lag): 0–4
- d (differencing): 0–1
- q (MA lag): 0–4
- Trials: `n_trials_test` in test mode, `n_trials` in full mode (see `model_config.yaml`)

In [ ]:
arima_ps = model_cfg['models']['arima'].get('param_search', {})
N_TRIALS_ARIMA = (arima_ps.get('n_trials_test', 20)
                  if base_cfg.get('test_mode', False)
                  else arima_ps.get('n_trials', 40))
arima_ss = arima_ps.get('search_space', {})

train_series = train_df[TARGET].dropna().values
val_series   = val_df[TARGET].dropna().values

print(f'ARIMA order search: {N_TRIALS_ARIMA} trials (mode: {MODE})')
print(f'Search space: p={arima_ss.get("p", [0,4])}, d={arima_ss.get("d", [0,1])}, q={arima_ss.get("q", [0,4])}')

def arima_objective(trial):
    p = trial.suggest_int('p', *arima_ss.get('p', [0, 4]))
    d = trial.suggest_int('d', *arima_ss.get('d', [0, 1]))
    q = trial.suggest_int('q', *arima_ss.get('q', [0, 4]))
    history = list(train_series)
    preds = []
    for actual in val_series:
        try:
            res = ARIMA(history, order=(p, d, q)).fit()
            preds.append(res.forecast(steps=1)[0])
        except Exception:
            preds.append(history[-1])
        history.append(actual)
    return mean_absolute_error(val_series, preds)

arima_study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
arima_study.optimize(arima_objective, n_trials=N_TRIALS_ARIMA, show_progress_bar=True)

bp = arima_study.best_params
BEST_ARIMA_ORDER = (bp['p'], bp['d'], bp['q'])
print(f'\nBest ARIMA order: {BEST_ARIMA_ORDER} (val MAE = {arima_study.best_value:.2f}K)')

# Save both search results to Drive
baseline_best = {
    'naive_lag1_weight': float(BEST_NAIVE_WEIGHT),
    'arima_order': list(BEST_ARIMA_ORDER),
    'naive_val_mae': best_naive['val_mae'],
    'arima_val_mae': round(arima_study.best_value, 2),
    'n_trials_arima': N_TRIALS_ARIMA,
}
baseline_params_path = METRICS_DIR / 'baseline_best_params.json'
with open(baseline_params_path, 'w') as f:
    json.dump(baseline_best, f, indent=2)
print(f'Best params saved: {baseline_params_path}')

## Model 1 — Naive Baseline

Predicts next month's payroll change = last observed payroll change (`payems_lag1`).

In [ ]:
print(f'=== Naive Baseline (lag1_weight={BEST_NAIVE_WEIGHT}) ===')
print(f'Prediction = {BEST_NAIVE_WEIGHT}*payems_lag1 + {1-BEST_NAIVE_WEIGHT:.1f}*payems_lag2')
print(f'Train: {train_df["forecast_date"].min().date()} to {train_df["forecast_date"].max().date()}')
print(f'Test : {test_df["forecast_date"].min().date()} to {test_df["forecast_date"].max().date()}')

def predict_naive(df, w):
    lag1 = df['payems_lag1'].fillna(0)
    lag2 = df['payems_lag2'].fillna(df['payems_lag1'].fillna(0))
    return (w * lag1 + (1 - w) * lag2).values

naive_val_pred  = predict_naive(val_df, BEST_NAIVE_WEIGHT)
naive_test_pred = predict_naive(test_df, BEST_NAIVE_WEIGHT)
naive_test_actual = test_df[TARGET].values
naive_test_dates  = test_df['forecast_date'].values
naive_val_actual  = val_df[TARGET].values

m_val  = compute_metrics(naive_val_actual,  naive_val_pred,  'naive', 'val')
m_test = compute_metrics(naive_test_actual, naive_test_pred, 'naive', 'test')
all_metrics.extend([m_val, m_test])

print(f'\nVal  — MAE={m_val["MAE"]:.1f}K, RMSE={m_val["RMSE"]:.1f}K, DA={m_val["DA"]:.2%}')
print(f'Test — MAE={m_test["MAE"]:.1f}K, RMSE={m_test["RMSE"]:.1f}K, DA={m_test["DA"]:.2%}')

naive_pred_df = show_prediction_sample(naive_test_dates, naive_test_actual, naive_test_pred, 'Naive')

## Model 2 — ARIMA

ARIMA(2,0,2) trained on the PAYEMS MoM change series (training set only).
Evaluated on the test set using a walk-forward (expanding window) approach.

In [ ]:
print(f'=== ARIMA{BEST_ARIMA_ORDER} (best order from search) ===')
print(f'Train period: {train_df["forecast_date"].min().date()} to {train_df["forecast_date"].max().date()}')

all_data = model_df[['forecast_date', TARGET]].dropna()
arima_predictions = []
arima_actuals = []
arima_dates = []

for _, row in test_df.iterrows():
    fd = row['forecast_date']
    history = all_data[all_data['forecast_date'] < fd][TARGET].values
    if len(history) < 20:
        continue
    try:
        res = ARIMA(history, order=BEST_ARIMA_ORDER).fit()
        forecast = res.forecast(steps=1)[0]
    except Exception:
        forecast = row['payems_lag1']
    arima_predictions.append(forecast)
    arima_actuals.append(row[TARGET])
    arima_dates.append(fd)

m_arima = compute_metrics(arima_actuals, arima_predictions, 'arima', 'test')
all_metrics.append(m_arima)

print(f'Test — MAE={m_arima["MAE"]:.1f}K, RMSE={m_arima["RMSE"]:.1f}K, DA={m_arima["DA"]:.2%}')

arima_pred_df = show_prediction_sample(arima_dates, arima_actuals, arima_predictions, 'ARIMA')

## Baseline comparison summary

In [ ]:
metrics_df = pd.DataFrame(all_metrics)
print('=== Baseline Metrics ===')
print(metrics_df.to_string(index=False))
print()
print('These are the benchmarks that XGBoost models must beat.')

## Save predictions and metrics

In [ ]:
PREDS_DIR = DRIVE_ROOT / 'outputs' / 'predictions'
METRICS_DIR = DRIVE_ROOT / 'outputs' / 'metrics'

# Save naive predictions
naive_out = pd.DataFrame({
    'forecast_date': naive_test_dates,
    'actual': naive_test_actual,
    'prediction': naive_test_pred,
    'model_name': 'naive',
})
naive_path = PREDS_DIR / 'predictions_naive.parquet'
naive_out.to_parquet(naive_path, index=False)

# Save ARIMA predictions
arima_out = pd.DataFrame({
    'forecast_date': arima_dates,
    'actual': arima_actuals,
    'prediction': arima_predictions,
    'model_name': 'arima',
})
arima_path = PREDS_DIR / 'predictions_arima.parquet'
arima_out.to_parquet(arima_path, index=False)

# Save metrics
metrics_path = METRICS_DIR / 'baseline_metrics.csv'
metrics_df.to_csv(metrics_path, index=False)

print(f'Naive predictions: {naive_path}')
print(f'ARIMA predictions: {arima_path}')
print(f'Baseline metrics : {metrics_path}')

## Manual Approval Gate

**Before approving:**
1. Naive MAE is in the expected range (50–300K for PAYEMS)
2. ARIMA predictions are not constant or nonsensical
3. Both models have >50% directional accuracy
4. COVID period predictions are visually inspected (extreme errors expected)

In [ ]:
APPROVE_THIS_STEP = False

if not APPROVE_THIS_STEP:
    raise RuntimeError(
        'STOP: Inspect the diagnostics above. '
        'If everything is correct, change APPROVE_THIS_STEP=True and rerun this cell.'
    )

In [ ]:
approval = {
    'approved': True,
    'notebook': '08_train_baseline_models.ipynb',
    'approved_at': datetime.datetime.utcnow().isoformat(),
    'input_artifacts': [str(MODEL_READY_PATH)],
    'output_artifacts': [str(naive_path), str(arima_path), str(metrics_path)],
    'row_counts': {'naive_test': int(len(naive_out)), 'arima_test': int(len(arima_out))},
    'warnings': [],
    'notes': f'Naive test MAE={m_test["MAE"]}, ARIMA test MAE={m_arima["MAE"]}'
}
ap_path = DRIVE_ROOT / 'approvals' / '08_baseline_models_approved.json'
with open(ap_path, 'w') as f:
    json.dump(approval, f, indent=2)
print(f'Approval saved: {ap_path}')
print('Notebook 08 complete. Proceed to 09_train_xgboost_models.ipynb')